# MIE Separator Benchmark v0.1
Laboratorio externo para comparar **HTDemucs** y **BS-RoFormer 6-stem** sobre un único audio completo.

Objetivo MIE: obtener automáticamente representaciones utilizables para **melodía (vocals)**, **armonía (piano/other)** y **bajo**, manteniendo Beat This como motor temporal independiente.

Este notebook no ejecuta melodía, armonía ni IA. Solo evalúa el cuello de botella de separación.

## 0. Runtime
En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** antes de comenzar.

In [ ]:
import os, sys, subprocess, json, shutil, glob, pathlib, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)

## 1. Instalar separadores

In [ ]:
!pip -q install -U demucs bs-roformer-infer soundfile
print('Instalación terminada.')

## 2. Subir audio completo
Sube **una sola mezcla original**. WAV, MP3 o M4A son suficientes para el benchmark.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Sube exactamente un archivo de audio.')
input_name = next(iter(uploaded.keys()))
input_path = os.path.abspath(input_name)
print('Audio:', input_path)

## 3. Normalizar entrada para BS-RoFormer
BS-RoFormer-Infer procesa WAV de carpeta. Se crea una copia PCM sin modificar el archivo original.

In [ ]:
os.makedirs('/content/mie_input', exist_ok=True)
wav_path = '/content/mie_input/source.wav'
subprocess.run(['ffmpeg','-y','-i',input_path,'-ar','44100','-ac','2',wav_path], check=True)
print(wav_path)

## 4. HTDemucs baseline
Salida esperada: vocals, drums, bass, other.

In [ ]:
demucs_out = '/content/mie_outputs/htdemucs'
os.makedirs(demucs_out, exist_ok=True)
cmd = [sys.executable, '-m', 'demucs', '-n', 'htdemucs', '--out', demucs_out, input_path]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('HTDemucs terminado.')

## 5. BS-RoFormer 6-stem
Usa el modelo recomendado del registro de `bs-roformer-infer`. La primera ejecución descarga automáticamente el checkpoint (~700 MB) y verifica su SHA-256. Salida esperada: vocals, drums, bass, guitar, piano, other e instrumental.

In [ ]:
bs_out = '/content/mie_outputs/bsroformer'
os.makedirs(bs_out, exist_ok=True)
cmd = ['bs-roformer-infer', '--input_folder', '/content/mie_input', '--store_dir', bs_out, '--device', 'cuda']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('BS-RoFormer terminado.')

## 6. Inventario de resultados
Revisa especialmente: **HTDemucs/other**, **BS-RoFormer/piano**, **BS-RoFormer/other**, además de vocals y bass.

In [ ]:
from pathlib import Path
rows=[]
for engine, root in [('htdemucs',demucs_out),('bsroformer',bs_out)]:
    for p in sorted(Path(root).rglob('*.wav')):
        rows.append({'engine':engine,'file':str(p),'mb':round(p.stat().st_size/1048576,2)})
import pandas as pd
display(pd.DataFrame(rows))

## 7. Manifest MIE
Registra exactamente qué separadores produjeron qué artefactos. Este manifiesto servirá para trazabilidad del benchmark.

In [ ]:
manifest = {
  'schema':'MIE-Separator-Benchmark-v0.1',
  'source_file':input_name,
  'engines':{
    'htdemucs':{'model':'htdemucs','expected':['vocals','drums','bass','other']},
    'bsroformer':{'model':'roformer-model-bs-roformer-sw-by-jarredou (package default)','expected':['vocals','drums','bass','guitar','piano','other','instrumental']}
  },
  'mie_evaluation_priority':['bsroformer/piano','bsroformer/other','htdemucs/other','bsroformer/vocals','htdemucs/vocals','bsroformer/bass','htdemucs/bass'],
  'files':rows
}
with open('/content/mie_outputs/MIE_SEPARATOR_BENCHMARK_v0_1.json','w') as f:
    json.dump(manifest,f,indent=2)
print(json.dumps(manifest,indent=2)[:4000])

## 8. Empaquetar y descargar
Descarga un ZIP único. Después escucharemos externamente los candidatos y solo traeremos al chat la decisión/JSON mínimo necesario.

In [ ]:
zip_base='/content/MIE_Separator_Benchmark_v0_1'
shutil.make_archive(zip_base,'zip','/content/mie_outputs')
print(zip_base+'.zip')
files.download(zip_base+'.zip')

## Criterio experimental posterior
No elegir el modelo por 'sonar más bonito'. Para MIE evaluaremos:
1. **melodía:** claridad de `vocals` para transcripción;
2. **armonía:** utilidad de `piano`/`other` para Motor↔IA↔Motor;
3. **bajo:** estabilidad de altura y registro;
4. **automatización:** ausencia de intervención manual por canción.

Beat/tactus no forma parte de este benchmark: Beat This v1.9.3 permanece como baseline independiente.